In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Build a Text Cleaning Pipeline

In [2]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer, PorterStemmer

# Download if first time
nltk.download('stopwords')
nltk.download('wordnet')

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()
stemmer = PorterStemmer()

def text_cleaning_pipeline(dataset, rule="lemmatize"):
    """
    Cleans text data:
    - lowercase
    - remove URLs
    - remove emojis
    - remove symbols
    - remove stopwords
    - lemmatize or stem
    """

    # Convert to lowercase
    data = dataset.lower()

    # Remove URLs
    data = re.sub(r"http\S+|www\S+|https\S+", '', data)

    # Remove emojis
    data = re.sub(r'[^\x00-\x7F]+', '', data)

    # Remove unwanted characters
    data = re.sub(r'[^a-zA-Z\s]', '', data)

    # Tokenize
    tokens = data.split()

    # Remove stopwords
    tokens = [word for word in tokens if word not in stop_words]

    # Lemmatize or Stem
    if rule == "lemmatize":
        tokens = [lemmatizer.lemmatize(word) for word in tokens]

    elif rule == "stem":
        tokens = [stemmer.stem(word) for word in tokens]

    else:
        print("Pick between lemmatize or stem")

    return " ".join(tokens)


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


Observation:

The `text_cleaning_pipeline()` function is used to preprocess raw text data into a clean and structured format suitable for machine learning tasks such as sentiment analysis. It first converts the entire text to lowercase to ensure consistency. Then, it removes URLs, emojis, punctuation, numbers, and other special characters so that only meaningful textual content remains. After that, the text is split into individual words through tokenization. Common stopwords like “the,” “is,” and “and” are removed because they do not contribute much to the meaning of the sentence. Finally, the function applies either lemmatization or stemming based on the selected option: lemmatization converts words to their dictionary form (e.g., “running” → “run”), while stemming reduces words to their root form by trimming suffixes. The processed words are then joined back into a single cleaned string, which can be used for model training.


# Text Classification using Machine Learning Models



1. **Load the Dataset**  with two columns text and label
   

2. **Text Cleaning and Tokenization**  
   Apply a text preprocessing pipeline to the `"text"` column. This should include:
   - Lowercasing the text  
   - Removing URLs, mentions, punctuation, and special characters  
   - Removing stopwords  
   - Tokenization (optional: stemming or lemmatization)
   - "Complete the above function"

3. **Train-Test Split**  
   Split the cleaned and tokenized dataset into **training** and **testing** sets using `train_test_split` from `sklearn.model_selection`.

4. **TF-IDF Vectorization**  
   Import and use the `TfidfVectorizer` from `sklearn.feature_extraction.text` to transform the training and testing texts into numerical feature vectors.

5. **Model Training and Evaluation**  
   Import **Logistic Regression** (or any machine learning model of your choice) from `sklearn.linear_model`. Train it on the TF-IDF-embedded training data, then evaluate it using the test set.  
   


Importing libraries

In [3]:
import pandas as pd
import re
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer, PorterStemmer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

# Download NLTK resources (run once)
nltk.download('stopwords')
nltk.download('wordnet')


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

Loading dataset

In [4]:
import os
print(os.listdir())


['.config', 'drive', 'sample_data']


In [5]:
print(os.listdir('/content/drive/MyDrive/AI And ML/Week8'))


['trum_tweet_sentiment_analysis.csv', 'trumptweets_small.csv', 'Week8 _Text _Pre-processing_2417514.ipynb', 'Week8_Text_Classification _2417514.ipynb']


Loading the dataset

In [8]:
df = pd.read_csv("/content/drive/MyDrive/AI And ML/Week8/trum_tweet_sentiment_analysis.csv")

In [9]:
# Show first 5 rows
print(df.head())

# Check required columns
print("\nColumns:", df.columns)



                                                text  Sentiment
0  RT @JohnLeguizamo: #trump not draining swamp b...          0
1  ICYMI: Hackers Rig FM Radio Stations To Play A...          0
2  Trump protests: LGBTQ rally in New York https:...          1
3  "Hi I'm Piers Morgan. David Beckham is awful b...          0
4  RT @GlennFranco68: Tech Firm Suing BuzzFeed fo...          0

Columns: Index(['text', 'Sentiment'], dtype='object')


In [10]:
df = df[['text', 'Sentiment']]
df.rename(columns={'Sentiment':'label'}, inplace=True)


In [11]:
# Keep only needed columns
df = df[['text', 'label']]

# Remove missing values
df.dropna(inplace=True)

print("\nDataset Shape:", df.shape)


Dataset Shape: (1850123, 2)


Text cleaning function

In [12]:

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()
stemmer = PorterStemmer()

def text_cleaning_pipeline(dataset, rule="lemmatize"):
    """
    Cleans text data:
    - lowercase
    - remove URLs
    - remove mentions
    - remove punctuation
    - remove special chars
    - remove stopwords
    - tokenize
    - lemmatize or stem
    """

    # Convert to lowercase
    data = dataset.lower()

    # Remove URLs
    data = re.sub(r"http\S+|www\S+|https\S+", '', data)

    # Remove mentions (@username)
    data = re.sub(r'@\w+', '', data)

    # Remove hashtags symbol only (#trump -> trump)
    data = re.sub(r'#', '', data)

    # Remove punctuation / special characters / numbers
    data = re.sub(r'[^a-zA-Z\s]', '', data)

    # Tokenization
    tokens = data.split()

    # Remove stopwords
    tokens = [word for word in tokens if word not in stop_words]

    # Lemmatization or Stemming
    if rule == "lemmatize":
        tokens = [lemmatizer.lemmatize(word) for word in tokens]

    elif rule == "stem":
        tokens = [stemmer.stem(word) for word in tokens]

    else:
        print("Pick between lemmatize or stem")

    return " ".join(tokens)

Observation:

The `text_cleaning_pipeline()` function is designed to clean and preprocess raw text data for machine learning tasks such as sentiment analysis. It first converts the input text to lowercase to ensure uniformity. It then removes URLs, user mentions (like @username), and the hashtag symbol while keeping the actual word. After that, it eliminates punctuation, numbers, and special characters so only meaningful alphabetic text remains. The text is then split into individual words (tokenization), and common stopwords such as “is,” “the,” and “and” are removed. Finally, depending on the chosen option, the function either applies lemmatization to convert words into their base dictionary form or stemming to reduce words to their root form. The cleaned words are then joined back into a single processed string, making the text ready for feature extraction and model training.


Apply cleaning

In [13]:
df['clean_text'] = df['text'].apply(lambda x: text_cleaning_pipeline(x, rule="lemmatize"))

print("\nCleaned Text Sample:")
print(df[['text', 'clean_text']].head())


Cleaned Text Sample:
                                                text  \
0  RT @JohnLeguizamo: #trump not draining swamp b...   
1  ICYMI: Hackers Rig FM Radio Stations To Play A...   
2  Trump protests: LGBTQ rally in New York https:...   
3  "Hi I'm Piers Morgan. David Beckham is awful b...   
4  RT @GlennFranco68: Tech Firm Suing BuzzFeed fo...   

                                          clean_text  
0  rt trump draining swamp taxpayer dollar trip a...  
1  icymi hacker rig fm radio station play antitru...  
2    trump protest lgbtq rally new york bbcworld via  
3  hi im pier morgan david beckham awful donald t...  
4  rt tech firm suing buzzfeed publishing unverif...  


Observation:

This code applies the text preprocessing function to the dataset and creates a new cleaned version of the text.
First, it takes the "text" column from the dataframe and applies the text_cleaning_pipeline() function to each row using .apply(). A lambda function is used to pass each individual text entry into the cleaning pipeline with the "lemmatize" option, meaning words will be converted to their base dictionary form. The result of this processing is stored in a new column called "clean_text" in the dataframe.
After cleaning all the text data, the code prints a message "Cleaned Text Sample:" and then displays the first few rows of both the original "text" column and the newly created "clean_text" column using .head(). This allows you to compare raw text with cleaned text and verify that preprocessing has been applied correctly.

Defining features and labels

In [14]:
X = df['clean_text']
y = df['label']

Training and testing split

In [15]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\nTraining Size:", X_train.shape[0])
print("Testing Size:", X_test.shape[0])


Training Size: 1480098
Testing Size: 370025


Observation:

This code splits the dataset into training and testing sets using `train_test_split()` from Scikit-learn. It takes the feature data `X` (cleaned tweet text) and the target labels `y` (sentiment classes) and divides them into two parts: 80% for training and 20% for testing, controlled by `test_size=0.2`. The `random_state=42` ensures the split is reproducible, meaning the same split will be generated every time the code runs. The `stratify=y` parameter is used to maintain the same proportion of each class (e.g., positive/negative sentiment) in both training and testing sets, which is important for balanced model learning. After splitting, it prints the number of samples in the training set and the testing set using `shape[0]`, allowing you to confirm how many data points are used for training and evaluation.


TF-IDF Vectorization

In [16]:
vectorizer = TfidfVectorizer(max_features=5000)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print("\nTF-IDF Shape:", X_train_tfidf.shape)


TF-IDF Shape: (1480098, 5000)


Observation:

This code converts the cleaned text data into numerical form using **TF-IDF (Term Frequency–Inverse Document Frequency)** so that a machine learning model can understand it.

First, a TfidfVectorizer is created with max_features=5000, which means it will only keep the 5000 most important words from the dataset based on their frequency and importance. Then, fit_transform() is applied on `X_train`, which learns the vocabulary from the training data and converts it into a TF-IDF numerical matrix. After that, `transform()` is used on `X_test`, which converts the test data using the same learned vocabulary without re-learning it (to avoid data leakage).

Finally, it prints the shape of the transformed training data, showing how many rows (tweets) and features (words) are present in the TF-IDF matrix. This step is important because machine learning models cannot work directly with text—they require numerical input.


Train Logistic Regression Model

In [17]:
model = LogisticRegression(max_iter=1000)

model.fit(X_train_tfidf, y_train)

LogisticRegression(max_iter=1000)

Observation:

This code creates and trains a **Logistic Regression model** for sentiment classification.

First, `LogisticRegression(max_iter=1000)` initializes the model, where `max_iter=1000` increases the number of iterations allowed for the optimization process to ensure the model fully converges (especially useful for large text datasets like TF-IDF features).

Then, `model.fit(X_train_tfidf, y_train)` trains the model using the TF-IDF-transformed training data (`X_train_tfidf`) and their corresponding sentiment labels (`y_train`). During this step, the model learns patterns between word usage in tweets and their sentiment classes, so it can later predict sentiment for new unseen tweets.


Prediction

In [18]:
y_pred = model.predict(X_test_tfidf)

Evaluation

In [19]:
print("\nAccuracy Score:", accuracy_score(y_test, y_pred))

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))


Accuracy Score: 0.9261752584284846

Classification Report:

              precision    recall  f1-score   support

           0       0.94      0.96      0.95    248842
           1       0.90      0.87      0.88    121183

    accuracy                           0.93    370025
   macro avg       0.92      0.91      0.92    370025
weighted avg       0.93      0.93      0.93    370025



Observation:

The accuracy score of 0.926 means that the model correctly predicted the sentiment of about 92.6% of all tweets, which indicates strong overall performance.

The classification report gives a deeper breakdown:

* For class 0 (usually one sentiment like negative or neutral), the model has very high performance with precision 0.94 and recall 0.96, meaning it correctly identifies most class 0 tweets and makes few mistakes when predicting them.

* For class 1 (the other sentiment class), the model also performs well with precision 0.90 and recall 0.87, though slightly lower than class 0, meaning it misses some positive cases or misclassifies a few.

* The F1-score (harmonic mean of precision and recall) is 0.95 for class 0 and 0.88 for class 1, showing balanced performance overall but slightly weaker on class 1.

* The macro average 0.92 treats both classes equally, while the weighted average 0.93 accounts for class imbalance and confirms overall strong performance.

